## Utility functions for visualization

In [41]:
# Below functions are used to visualize the grid data in a more human-readable format
# imported from skeleton/utils.py

from pathlib import Path

import numpy as np

from rich.console import Console
from rich.text import Text
from typing import List

color_map = {
    0: "black",
    1: "red",
    2: "green",
    3: "yellow",
    4: "blue",
    5: "magenta",
    6: "cyan",
    7: "white",
    8: "bright_red",
    9: "bright_green",
}

console = Console(force_jupyter=True)

def make_rich_lines(grid: List[List[int]]) -> List[Text]:
    lines = []
    for row in grid:
        visual = Text()
        for cell in row:
            color = color_map.get(cell, "white")
            visual.append("  ", style=f"on {color}")
        raw = Text("  " + str(row))
        visual.append(raw)
        lines.append(visual)
    return lines

def render_grid(grid: List[List[int]]):
    lines = make_rich_lines(grid)
    console.print(Text("\n").join(lines)) # updated to print lines sequentially in Jupyter


## Load Dataset

In [4]:
import os

from transformers import set_seed
import json
import pandas as pd
import numpy as np

In [7]:
def load_data(base_dir):
    '''
    Load data from the specified directory and return a DataFrame.
    '''

    filenames = os.listdir(base_dir) # 파일명에 확장자 포함
    data_files = [os.path.join(base_dir, p) for p in filenames if ".json" in p]

    dataset = []
    for fn in data_files:
        with open(fn) as fp:
            data = json.load(fp)
        dataset.append(data)

    filenames = [fn.split(".")[0] for fn in filenames] # 확장자 제거
    data = []
    MAX_LEN = 1000 # train dataset의 최대 길이. 필요에 따라 조정 가능
    rng = np.random.default_rng(42)

    N = len(dataset)

    while len(data) < MAX_LEN:
        task_idx = rng.integers(0, N) # 랜덤으로 task 선택
        task = dataset[task_idx]
        file_name = filenames[task_idx]

        n_task = len(task)
        grids_idx =  rng.choice(n_task, size=4, replace=True) # 앞서 추출한 task에서 랜덤으로 4개의 grid 선택
        train_grids = [task[i] for i in grids_idx[:3]] # 3개는 train data로 사용
        test_grids = [task[i] for i in grids_idx[3:]] # 1개는 test data로 사용

        test_inputs = [{'input': grid['input']} for grid in test_grids]
        test_outputs = [grid['output'] for grid in test_grids]
        test_outputs_transformed = [{'output': grid} for grid in test_outputs]
        combined_tests = []
        for test_input, test_output in zip(test_inputs, test_outputs_transformed):
            combined_tests.append({'input': test_input['input'], 'output': test_output['output']})

        data.append({
            'task': file_name,
            'train': train_grids,
            'test_input': test_inputs,
            'test_output': test_outputs,
            'test': combined_tests,
        })

    df = pd.DataFrame(data)
    return df

In [6]:
token = os.environ.get("HF_TOKEN", None)
from arc import ARCSolver

solver = ARCSolver(token=token)

set_seed(1234567890)

data_path = "/workspace/dataset"
val_size = 0.1

dataset = load_data(data_path)

# from datasets import Dataset
# train_dataset = Dataset.from_pandas(dataset).shuffle(42).select(range(10))

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## EDA

In [ ]:
dataset.head()

,task,train,test_input,test_output,test
0,a2fd1cf0,"[{'input': [[7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7...","[{'input': [[1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1...","[[[1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1,...","[{'input': [[1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1..."
1,4938f0c2,"[{'input': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1...","[{'input': [[0, 0, 0, 0, 0, 0, 0, 0, 0], [4, 4...","[[[0, 0, 0, 0, 0, 0, 0, 0, 0], [4, 4, 4, 4, 0,...","[{'input': [[0, 0, 0, 0, 0, 0, 0, 0, 0], [4, 4..."
2,6455b5f5,"[{'input': [[5, 7, 5, 5, 7, 5, 5, 5, 7, 5], [7...","[{'input': [[7, 7, 5, 7, 7, 7, 5, 7], [7, 7, 5...","[[[7, 7, 5, 7, 7, 7, 5, 7], [7, 7, 5, 7, 7, 7,...","[{'input': [[7, 7, 5, 7, 7, 7, 5, 7], [7, 7, 5..."
3,d631b094,"[{'input': [[0, 0, 0, 0], [0, 9, 9, 9], [0, 0,...","[{'input': [[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], ...","[[[3, 3, 3]]]","[{'input': [[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], ..."
4,b1948b0a,"[{'input': [[4, 6, 6], [0, 6, 3], [0, 6, 4], [...","[{'input': [[6, 6, 9, 5, 8, 8, 1, 6, 6, 7], [7...","[[[2, 2, 9, 5, 8, 8, 1, 2, 2, 7], [7, 2, 2, 7,...","[{'input': [[6, 6, 9, 5, 8, 8, 1, 6, 6, 7], [7..."


Below is the visualization code for each task. You may visualize the task you want by setting `task_idx` as you like.

In [60]:
task_idx = 999

print(f"[Task #{task_idx} ({dataset['task'][task_idx]}) in dataset]\n")

print("train example #0")
render_grid(dataset['train'][task_idx][0]['input'])
render_grid(dataset['train'][task_idx][0]['output'])

print("train example #1")
render_grid(dataset['train'][task_idx][1]['input'])
render_grid(dataset['train'][task_idx][1]['output'])

print("train example #2")
render_grid(df['train'][task_idx][2]['input'])
render_grid(df['train'][task_idx][2]['output'])

print("test input")
render_grid(df['test_input'][task_idx][0]['input'])

print("test output")
render_grid(df['test_output'][task_idx][0])

[Task #999 (5521c0d9) in dataset]

train example #0


                    [0, 0, 0, 0, 0, 0, 0, 0, 0]
                    [1, 1, 1, 1, 1, 1, 1, 0, 0]
                    [1, 1, 1, 1, 1, 1, 1, 0, 0]
                    [0, 0, 0, 0, 0, 0, 0, 0, 0]
                    [0, 0, 0, 0, 0, 0, 0, 0, 0]
                    [6, 6, 6, 0, 0, 0, 0, 0, 0]
                    [6, 6, 6, 0, 0, 0, 0, 0, 0]
                    [0, 0, 0, 0, 0, 0, 0, 0, 0]
                    [0, 0, 0, 0, 0, 0, 0, 0, 0]
                    [0, 0, 0, 0, 0, 0, 0, 0, 0]

                    [0, 0, 0, 0, 0, 0, 0, 0, 0]
                    [0, 0, 0, 0, 0, 0, 0, 1, 1]
                    [0, 0, 0, 0, 0, 0, 0, 1, 1]
                    [0, 0, 0, 0, 0, 0, 0, 0, 0]
                    [0, 0, 0, 0, 0, 0, 0, 0, 0]
                    [0, 0, 0, 6, 6, 6, 0, 0, 0]
                    [0, 0, 0, 6, 6, 6, 0, 0, 0]
                    [0, 0, 0, 0, 0, 0, 0, 0, 0]
                    [0, 0, 0, 0, 0, 0, 0, 0, 0]
                    [0, 0, 0, 0, 0, 0, 0, 0, 0]

train example #1


                      [3, 3, 3, 3, 3, 3, 3, 3, 3, 3]
                      [3, 3, 3, 3, 3, 3, 3, 3, 3, 3]
                      [3, 3, 3, 3, 3, 3, 3, 3, 3, 4]
                      [3, 3, 3, 3, 3, 3, 3, 3, 3, 4]
                      [3, 3, 3, 3, 3, 3, 3, 3, 3, 3]
                      [3, 3, 3, 3, 3, 3, 3, 3, 3, 3]
                      [3, 3, 3, 3, 3, 3, 3, 3, 3, 3]

...

                  [1, 1, 1, 1, 1, 1, 1, 1]
                  [1, 0, 0, 1, 1, 1, 1, 1]
                  [1, 1, 1, 1, 1, 1, 1, 1]
                  [1, 1, 1, 1, 1, 1, 1, 1]
                  [1, 1, 1, 1, 1, 1, 1, 1]
                  [1, 1, 1, 1, 1, 1, 1, 1]
                  [1, 1, 1, 1, 1, 5, 5, 1]

test input


                      [2, 1, 1, 1, 2, 0, 0, 3, 3, 2]
                      [2, 1, 1, 1, 2, 2, 2, 2, 2, 2]
                      [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
                      [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
                      [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
                      [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
                      [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
                      [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
                      [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]

test output


                      [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
                      [2, 2, 2, 2, 2, 0, 0, 3, 3, 2]
                      [2, 1, 1, 1, 2, 2, 2, 2, 2, 2]
                      [2, 1, 1, 1, 2, 2, 2, 2, 2, 2]
                      [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
                      [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
                      [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
                      [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
                      [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]

## Dataset

In [2]:
def load_data(base_dir):
    filenames = os.listdir(base_dir)
    data_files = [os.path.join(base_dir, p) for p in filenames if ".json" in p]

    dataset = []
    for fn in data_files:
        with open(fn) as fp:
            data = json.load(fp)
        dataset.append(data)

    filenames = [fn.split(".")[0] for fn in filenames]
    data = []
    MAX_LEN = 10
    rng = np.random.default_rng(42)

    N = len(dataset)

    while len(data) < MAX_LEN:
        task_idx = rng.integers(0, N)
        task = dataset[task_idx]
        file_name = filenames[task_idx]

        n_task = len(task)
        grids_idx =  rng.choice(n_task, size=4, replace=True)
        train_grids = [task[i] for i in grids_idx[:3]]
        test_grids = [task[i] for i in grids_idx[3:]]

        test_inputs = [{'input': grid['input']} for grid in test_grids]
        test_outputs = [grid['output'] for grid in test_grids]
        test_outputs_transformed = [{'output': grid} for grid in test_outputs]
        combined_tests = []
        for test_input, test_output in zip(test_inputs, test_outputs_transformed):
            combined_tests.append({'input': test_input['input'], 'output': test_output['output']})

        data.append({
            'task': file_name,
            'train': train_grids,
            'test_input': test_inputs,
            'test_output': test_outputs,
            'test': combined_tests,
        })

    df = pd.DataFrame(data)
    return df

`Dataset.from_pandas` function converts the pandas dataframe above into the huggingface dataset. Unlike `df`, `dataset` is an array so that we may iteratively address each data.

In [88]:
from datasets import Dataset
dataset = Dataset.from_pandas(df).shuffle(42)
dataset

Dataset({
    features: ['task', 'train', 'test_input', 'test_output', 'test'],
    num_rows: 1000
})

Note the position of `task_idx`.

In [ ]:
task_idx = 999

print(f"[Task #{task_idx} ({dataset[task_idx]['task']}) in dataset]\n")

print("train example #0")
render_grid(dataset[task_idx]['train'][0]['input'])
render_grid(dataset[task_idx]['train'][0]['output'])

print("train example #1")
render_grid(dataset[task_idx]['train'][1]['input'])
render_grid(dataset[task_idx]['train'][1]['output'])

print("train example #2")
render_grid(dataset[task_idx]['train'][2]['input'])
render_grid(dataset[task_idx]['train'][2]['output'])

print("test input")
render_grid(dataset[task_idx]['test_input'][0]['input'])
# or
# render_grid(dataset[task_idx]['test'][0]['input'])

print("test output")
render_grid(dataset[task_idx]['test_output'][0])
# or
# render_grid(dataset[task_idx]['test'][0]['output'])

[Task #999 (67a423a3) in dataset]

train example #0


                  [7, 8, 7, 7, 7, 7, 7, 7]
                  [0, 8, 0, 0, 0, 0, 0, 0]
                  [0, 8, 0, 0, 0, 0, 0, 0]
                  [7, 8, 7, 7, 7, 7, 7, 7]
                  [7, 8, 7, 7, 7, 7, 7, 7]
                  [7, 8, 7, 7, 7, 7, 7, 7]
                  [7, 8, 7, 7, 7, 7, 7, 7]

                  [4, 4, 4, 7, 7, 7, 7, 7]
                  [4, 8, 4, 0, 0, 0, 0, 0]
                  [4, 8, 4, 0, 0, 0, 0, 0]
                  [4, 4, 4, 7, 7, 7, 7, 7]
                  [7, 8, 7, 7, 7, 7, 7, 7]
                  [7, 8, 7, 7, 7, 7, 7, 7]
                  [7, 8, 7, 7, 7, 7, 7, 7]

train example #1


                [3, 3, 7, 7, 3, 3, 3]
                [3, 3, 7, 7, 3, 3, 3]
                [3, 3, 7, 7, 3, 3, 3]
                [5, 5, 7, 7, 5, 5, 5]
                [3, 3, 7, 7, 3, 3, 3]
                [3, 3, 7, 7, 3, 3, 3]
                [3, 3, 7, 7, 3, 3, 3]
                [3, 3, 7, 7, 3, 3, 3]

...

                      [0, 0, 0, 0, 0, 8, 8, 0, 0, 0]
                      [7, 7, 7, 7, 7, 8, 8, 7, 7, 7]
                      [0, 0, 0, 0, 0, 8, 8, 0, 0, 0]
                      [0, 0, 0, 0, 0, 8, 8, 0, 0, 0]
                      [0, 0, 0, 0, 0, 8, 8, 0, 0, 0]
                      [0, 0, 0, 0, 0, 8, 8, 0, 0, 0]
                      [0, 0, 0, 0, 0, 8, 8, 0, 0, 0]

test output


                      [0, 0, 0, 0, 4, 4, 4, 4, 0, 0]
                      [7, 7, 7, 7, 4, 8, 8, 4, 7, 7]
                      [0, 0, 0, 0, 4, 4, 4, 4, 0, 0]
                      [0, 0, 0, 0, 0, 8, 8, 0, 0, 0]
                      [0, 0, 0, 0, 0, 8, 8, 0, 0, 0]
                      [0, 0, 0, 0, 0, 8, 8, 0, 0, 0]
                      [0, 0, 0, 0, 0, 8, 8, 0, 0, 0]

                      [0, 0, 0, 0, 0, 8, 8, 0, 0, 0]
                      [7, 7, 7, 7, 7, 8, 8, 7, 7, 7]
                      [0, 0, 0, 0, 0, 8, 8, 0, 0, 0]
                      [0, 0, 0, 0, 0, 8, 8, 0, 0, 0]
                      [0, 0, 0, 0, 0, 8, 8, 0, 0, 0]
                      [0, 0, 0, 0, 0, 8, 8, 0, 0, 0]
                      [0, 0, 0, 0, 0, 8, 8, 0, 0, 0]

                      [0, 0, 0, 0, 4, 4, 4, 4, 0, 0]
                      [7, 7, 7, 7, 4, 8, 8, 4, 7, 7]
                      [0, 0, 0, 0, 4, 4, 4, 4, 0, 0]
                      [0, 0, 0, 0, 0, 8, 8, 0, 0, 0]
                      [0, 0, 0, 0, 0, 8, 8, 0, 0, 0]
                      [0, 0, 0, 0, 0, 8, 8, 0, 0, 0]
                      [0, 0, 0, 0, 0, 8, 8, 0, 0, 0]

In [90]:
train_val_split = dataset.train_test_split(test_size=0.1, seed=42)

In [91]:
train_dataset = train_val_split['train']
train_dataset

Dataset({
    features: ['task', 'train', 'test_input', 'test_output', 'test'],
    num_rows: 900
})

In [92]:
val_dataset = train_val_split['test']
val_dataset

Dataset({
    features: ['task', 'train', 'test_input', 'test_output', 'test'],
    num_rows: 100
})

## Format Dataset

In [16]:
N_data = 4
data_path = "/workspace/dataset"
df = load_data(data_path)

from datasets import Dataset
dataset = Dataset.from_pandas(df).shuffle(42).select(range(N_data))

In [17]:
def _format_data(datapoint, is_train=False):
    prompt = solver.format_prompt(datapoint, is_train)
    input_text = solver.tokenizer.decode(prompt['input_ids'])
    return {'text': input_text}

dataset = dataset.map(lambda x: _format_data(x, is_train=True), remove_columns=dataset.column_names)
print(dataset[0])

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

{'text': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\nYou are a puzzle solving wizard. You are given a puzzle from the abstraction and reasoning corpus developed by Francois Chollet.<|start_header_id|>user<|end_header_id|>\nHere are the example input and output pairs from which you should learn the underlying rule to later predict the output for the given test input:\n----------------------------------------\ninput:\n5444\n4474\n4444\n4444\noutput:\n54444444\n45744444\n44574444\n44457444\n44445744\n44444574\n44444457\n44444445\ninput:\n76661\n66666\n66666\n66666\noutput:\n7666166666\n6766616666\n6676661666\n6667666166\n6666766616\n6666676661\n6666667666\n6666666766\ninput:\n8818\n8888\n8879\n8888\n2288\noutput:\n88188888\n88818888\n88791888\n88879188\n22887918\n82288791\n88228879\n88822887\n88882288\n88888228\n\n----------------------------------------\nNow, solve the following puzzle based on its input grid by applying the rules you have learned from the training data